# Solution: Representing Time-Series Data in pandas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
US = {"Orange": "#EE7622"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2

In [ ]:
DATA_PATH = "../data/bikeshare_rides.csv"

The operations team wants to know whether weekend rides are consistently higher, and whether that pattern is seasonal (stronger in summer) or structural (consistent year-round). Complete the analysis below.

In [ ]:
# Load the CSV, parse the date column, and set it as the index
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.set_index("date")

# Remove any duplicate dates
df = df[~df.index.duplicated(keep="first")]

# Create a month column (as a period) and a day-of-week column
df["month"] = df.index.to_period("M")
df["dayofweek"] = df.index.dayofweek

# Compute average rides by month and day of week
pivot = df.groupby(["month", "dayofweek"])["rides"].mean().unstack()
print(pivot.head())

In [ ]:
# Compute weekend (Saturday=5, Sunday=6) and weekday averages per month
weekend = pivot[[5, 6]].mean(axis=1)
weekday = pivot[[0, 1, 2, 3, 4]].mean(axis=1)

# Compute the ratio of weekend to weekday rides
ratio = weekend / weekday
print(ratio.head())

# Return True if the standard deviation of the ratio exceeds this threshold
is_seasonal = ratio.std() > 0.1
print(f"Seasonal variation: {is_seasonal}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
pivot.T.plot(ax=ax, color=[UB["Brand Blue"], UB["Medium Blue"], UN["Dark Gray"], US["Orange"]])
ax.set_title("Average Rides by Day of Week (Monthly)", fontsize=18, fontweight="bold")
ax.set_ylabel("Average Rides", fontsize=16)
ax.tick_params(axis="both", labelsize=14)
ax.legend(title="Month", fontsize=12)
plt.tight_layout()
plt.show()